In [1]:
# Install LangChain and Ollama integration
!pip install -q langchain langchain-ollama


[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import ChatOllama to communicate with a locally running LLM
from langchain_ollama import ChatOllama

### Initialize the LLM

In [3]:
# Create an LLM object using the Ollama model

llm = ChatOllama(
    model="llama3",      # Local model downloaded using Ollama
    temperature=0         # Lower temperature = more deterministic responses
)

### **LLM CALL**

In [4]:
# Invoke (send) a prompt to the LLM
response = llm.invoke(
    "What is LLM? Answer in 10 words."
)

# Print only the text/content returned by the model
print(response.content)

Large Language Model: AI-powered text generator for natural language processing.


In [5]:
# Import the requests library to make HTTP requests
import requests

# Send a GET request to the Exchange Rate API
# This returns the latest exchange rates with USD as the base currency
rate = requests.get("https://open.er-api.com/v6/latest/USD").json()

# Extract the INR exchange rate from the JSON response
# 'rates' is a dictionary containing all supported currencies
current_rate = rate["rates"]["INR"]

# Print the USD to INR exchange rate
print(current_rate)

95.766882


In [6]:
# Send a prompt to the LLM using the latest exchange rate
response = llm.invoke(
    f"""
    Current USD to INR exchange rate is {current_rate}.

    Explain this exchange rate in one simple sentence.
    """
)

# Print only the text generated by the LLM
print(response.content)

As of now, 1 US Dollar (USD) can be exchanged for approximately 95 Indian Rupees (INR).


In [7]:
# Install LangChain, Ollama integration, and DuckDuckGo search
!pip install -q langchain langchain-ollama langchain-community duckduckgo-search


[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# Import the DuckDuckGo search tool
from langchain_community.tools import DuckDuckGoSearchResults

# Create an instance of the DuckDuckGo search tool
search = DuckDuckGoSearchResults()
# Search for the latest AI news
news = search.invoke("top AI news today")
# Ask the LLM to summarize the search results
response = llm.invoke(
    f"""
    Summarize the following AI news in simple language:

    {news}
    """
)
# Print the generated summary
print(response.content)

C:\Users\anubh\AppData\Local\Temp\ipykernel_16832\389323670.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchResults


Here are the summaries of the AI news in simple language:

**AI Weekly**

* A newsletter that delivers AI news three times a week.
* It covers funding, models, regulation, and applications related to artificial intelligence.
* Over 50,000 professionals read it since 2015.

**AI News Today: Daily Top 5 AI Stories (Updated Daily)**

* A daily briefing on the top AI stories, including breakthroughs and model releases.
* It's a quick 5-minute update on the most important AI news.

**Google News - News about Gemini • AI**

* This week's AI news roundup covers several topics:
	+ Lovable's deal with Google Cloud
	+ Railway's $100 million raise
	+ Walmart's AI cost realities
	+ E.ON's grid modernization using SAP AI

**AI News Today: The Latest AI News, Insights and Reports**

* A daily update on the latest AI news, insights, and reports.
* It covers top stories in the field of artificial intelligence.

These summaries are designed to give you a quick overview of each AI news source.


In [9]:
pip install -U ddgs

  Attempting uninstall: ddgs
    Found existing installation: ddgs 9.14.4
    Uninstalling ddgs-9.14.4:
      Successfully uninstalled ddgs-9.14.4
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
# Import the requests library to make HTTP requests
import requests

# Open-Meteo provides free weather APIs without requiring an API key.
# Latitude and longitude below correspond to New Delhi.
url = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=19.9975"
    "&longitude=73.7898"
    "&current=temperature_2m"
)

# Send a GET request to the weather API.
# Convert the JSON response into a Python dictionary.
weather_data = requests.get(url).json()

# Extract the current temperature (2 meters above ground level)
temperature = weather_data["current"]["temperature_2m"]

# Send the temperature to the LLM for explanation
response = llm.invoke(
    f"""
    Current temperature is {temperature}°C.

    Describe the weather in one simple sentence.
    """
)

# Print only the generated response
print(response.content)

The weather is warm and pleasant, with a current temperature of 27.2°C.


In [11]:
# Used to call REST APIs
import requests

# Used to convert a normal Python function into a LangChain Tool
from langchain_core.tools import tool

In [12]:
@tool
def get_temperature():
    """
    Get the current temperature in Nashik.
    """

    # Weather API URL
    url = (
        "https://api.open-meteo.com/v1/forecast"
        "?latitude=19.9975"
        "&longitude=73.7898"
        "&current=temperature_2m"
    )

    # Fetch the weather data
    weather_data = requests.get(url).json()

    # Return only the current temperature as a string
    return str(weather_data["current"]["temperature_2m"])

@tool
def get_usd_inr() -> str:
    """
    Returns the latest USD to INR exchange rate.
    Use this whenever the user asks about currency conversion.
    """

    url = "https://open.er-api.com/v6/latest/USD"

    response = requests.get(url)

    exchange_data = response.json()

    exchange_rate = exchange_data["rates"]["INR"]

    return str(exchange_rate)


@tool
def get_ai_news() -> str:
    """
    Returns latest AI news from DuckDuckGo search.
    """

    return search.invoke("latest AI news")

In [13]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:7b",   # Tool-calling capable model
    temperature=0
)

In [14]:
# Bind (attach) the tools to the LLM
tool_llm = llm.bind_tools(
    [
        get_temperature,
        get_usd_inr,
        get_ai_news
    ]
)



In [15]:
# User's question
question = "What is the temperature outside?"

# Send the question to the LLM that has tools attached
response = tool_llm.invoke(question)

# Print the tool calls suggested by the LLM
print(response.tool_calls)

[{'name': 'get_temperature', 'args': {'city': 'Nashik'}, 'id': '1adef1f0-d332-4a14-9d42-7d99540520a8', 'type': 'tool_call'}]


In [17]:
from langchain.agents import create_agent

# Create an agent using our Ollama LLM and existing tools
agent = create_agent(
    model=llm,
    tools=[
        get_temperature,
        get_usd_inr,
        get_ai_news
    ]
)

In [18]:
# User's question
question = "What is today's USD to INR exchange rate?"

# Run the agent
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    }
)

print(response)

{'messages': [HumanMessage(content="What is today's USD to INR exchange rate?", additional_kwargs={}, response_metadata={}, id='0cf65389-0d7a-44ab-91e7-d88560b948a5'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-08-19T09:06:02.2200684Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5977143300, 'load_duration': 5061922300, 'prompt_eval_count': 232, 'prompt_eval_duration': 209775200, 'eval_count': 19, 'eval_duration': 674618400, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--01a01945-2211-7f81-8fc1-d4efc4d15523-0', tool_calls=[{'name': 'get_usd_inr', 'args': {}, 'id': 'cdaa5721-ce8f-44b6-94d6-4d766d431b7a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 232, 'output_tokens': 19, 'total_tokens': 251}), ToolMessage(content='95.766882', name='get_usd_inr', id='2e157cf0-fbb4-4a3d-8f18-7d920087315e', tool_call_id='cdaa5721-ce8f-44b6-94d6-4d766d431b7a')

In [19]:
print(response["messages"][-1].content)

Today's USD to INR exchange rate is approximately 95.77 (rounded to two decimal places).


In [20]:
#Memory concept
response = llm.invoke(
    "my special please."
)

print(response.content)

Of course! Could you please provide more details about what you'd like me to assist you with? Whether it's information, a specific question, or something else, I'm here to help!


In [21]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage
)

# Send system instruction + user message together
response = llm.invoke(
    [
        # Defines how the AI should behave
        SystemMessage(
            content="You are a polite chai seller."
        ),

        # User's actual message with context
        HumanMessage(
            content="I am your regular customer. I love your special ginger tea. My special please."
        )
    ]
)

# Print LLM response
print(response.content)

Of course, right away! For you, it's a special ginger tea with just the right amount of warmth and spice. Coming up just for you! Enjoy your favorite brew!


In [22]:
# First call: Tell the LLM your favorite tea
print(
    llm.invoke(
        "My favorite tea is ginger tea."
    ).content
)

# Second call: Ask the LLM about the information
# given in the previous call
print(
    llm.invoke(
        "What is my favorite tea?"
    ).content
)

That's great! Ginger tea can be a delightful and warming beverage, especially during colder months or when you're feeling under the weather. It has natural properties that can help soothe an upset stomach and provide relief from nausea. Would you like to know more about how to make ginger tea or its potential health benefits?
To determine your favorite tea, I would need more information about you and your preferences. However, since I don't have any personal data about you, could you tell me what types of teas you enjoy or which ones you've tried before? This can include black tea, green tea, herbal tea, oolong tea, white tea, or any specific flavors like mint, chamomile, or jasmine. Knowing this will help narrow down the possibilities!


In [23]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Create in-memory short-term memory
memory = InMemorySaver()

# Create agent with Ollama model + memory
conversation = create_agent(
    model=llm,
    tools=[],
    checkpointer=memory
)

# Same thread_id = same conversation memory
config = {
    "configurable": {
        "thread_id": "tea_conversation"
    }
}

# First message
response1 = conversation.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My favorite tea is ginger tea."
            }
        ]
    },
    config=config
)

print(response1["messages"][-1].content)

# Second message
response2 = conversation.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is my favorite tea?"
            }
        ]
    },
    config=config
)

print(response2["messages"][-1].content)

That's great! Ginger tea can be a delightful and warming beverage, especially during colder months or when you're feeling under the weather. It has natural properties that can help soothe an upset stomach and provide relief from nausea. Would you like to know more about how to make ginger tea or its potential health benefits?
Based on your statement, your favorite tea is ginger tea. If you have any specific questions about making it, the benefits of ginger tea, or if you'd like some variations or recipes, feel free to ask!


In [24]:
# First independent LLM call
print(
    llm.invoke(
        "Explain chai simply for a beginner in 1 line"
    ).content
)


Chai is a spicy Indian tea made with black tea, milk, and a blend of spices like cinnamon, cardamom, and ginger.


In [25]:
# Second independent LLM call
print(
    llm.invoke(
        "Explain LangChain simply for a beginner in 1 line"
    ).content
)

LangChain is a framework that helps build applications using language models to chain together sequences of interactions and tasks.


In [26]:
from langchain_core.prompts import ChatPromptTemplate

# Create a reusable prompt template
# {topic} is a dynamic variable/place-holder
prompt = ChatPromptTemplate.from_template(
    "Explain {topic} simply for a beginner in 1 line"
)

# Create a chain:
# Prompt output → passed to → LLM
chain = prompt | llm


# Run same chain with different inputs
print(
    chain.invoke({"topic": "chai"}).content
)

print(
    chain.invoke({"topic": "LangChain"}).content
)

Chai is a spicy Indian tea made with black tea, milk, and a blend of spices like cinnamon, cardamom, and ginger.
LangChain is a framework that helps build applications using language models to chain together sequences of interactions and tasks.


In [27]:
from langchain_core.prompts import ChatPromptTemplate

# Create a prompt template with a dynamic variable {topic}
prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in 1 line"
)

# Fill {topic} with "LangChain"
# Then convert the generated prompt into chat messages
messages = prompt.invoke(
    {"topic": "LangChain"}
).to_messages()

# See the actual messages generated by the prompt template
print(messages)

# Send those messages to the Ollama LLM
response = llm.invoke(messages)

# Print only the generated text
print(response.content)

[HumanMessage(content='Explain LangChain in 1 line', additional_kwargs={}, response_metadata={})]
LangChain is a framework for building and chaining together language models to solve complex natural language processing tasks.


In [28]:
print(
    llm.invoke(
        "Write a title and a 3-line blog about AI in one response."
    ).content
)

Title: "The Future is Now: How Artificial Intelligence Is Transforming Our World"

Blog snippet: Embracing the power of artificial intelligence, we're witnessing unprecedented advancements that are reshaping industries from healthcare to entertainment. With smarter algorithms and increased data processing capabilities, AI is not just a future prospect but an integral part of our present, driving innovation and efficiency like never before.


In [29]:
from langchain_core.prompts import ChatPromptTemplate


# -----------------------------
# STEP 1: Create Prompt Templates
# -----------------------------

# First prompt generates a title
title_prompt = ChatPromptTemplate.from_template(
    "Write a short title about {topic}."
)

# Second prompt generates a blog using the title
# produced by the first chain
blog_prompt = ChatPromptTemplate.from_template(
    "Write a 3-line blog using this title: {title}"
)


# -----------------------------
# STEP 2: Create Chains
# -----------------------------

# Topic → Title Prompt → Ollama
title_chain = title_prompt | llm

# Title → Blog Prompt → Ollama
blog_chain = blog_prompt | llm


# -----------------------------
# STEP 3: Execute First Chain
# -----------------------------

title = title_chain.invoke(
    {"topic": "AI"}
).content


# -----------------------------
# STEP 4: Pass Output to Next Chain
# -----------------------------

blog = blog_chain.invoke(
    {"title": title}
).content


# -----------------------------
# STEP 5: Print Results
# -----------------------------

print(title)
print(blog)

"AI Revolution: Shaping the Future of Technology"
AI Revolution: Shaping the Future of Technology  
In the era of rapid technological advancement, artificial intelligence (AI) stands at the forefront, transforming industries and paving the way for unprecedented innovations. From healthcare to autonomous vehicles, AI's impact is reshaping our world in profound ways.

As we stand on the brink of this revolution, embracing AI means not just adapting to change but actively shaping a future where technology serves humanity more effectively than ever before.


## CHAINS

In [32]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


# -----------------------------------
# 1. Prompt for generating the title
# -----------------------------------

title_prompt = ChatPromptTemplate.from_template(
    "Write a short title about {topic}"
)

# topic -> prompt -> Ollama -> title text
title_chain = title_prompt | llm


# -----------------------------------
# 2. Prompt for generating the blog
# -----------------------------------

blog_prompt = ChatPromptTemplate.from_template(
    "Write a 3-line blog using this title: {title}"
)

# title -> prompt -> Ollama -> blog text
blog_chain = blog_prompt | llm


# -----------------------------------
# 3. Create Sequential Chain
# -----------------------------------

chain = (
    # Keep original input and generate title
    RunnablePassthrough.assign(
        title=lambda x: title_chain.invoke(
            {"topic": x["topic"]}
        ).content
    )

    # Use generated title to generate blog
    | RunnablePassthrough.assign(
        blog=lambda x: blog_chain.invoke(
            {"title": x["title"]}
        ).content
    )
)


# -----------------------------------
# 4. Run the complete chain
# -----------------------------------

result = chain.invoke({
    "topic": "LLM"
})


# -----------------------------------
# 5. Print outputs
# -----------------------------------

print("TITLE:")
print(result["title"])

print("\nBLOG:")
print(result["blog"])

TITLE:
"Large Language Models: Transforming Text Processing"

BLOG:
"Large Language Models: Transforming Text Processing" marks a new era in text analysis and generation. These models, with their vast training data and sophisticated algorithms, are revolutionizing how we interact with digital text, offering unprecedented accuracy and versatility. From enhancing search engines to powering creative writing tools, the impact is profound and far-reaching.


In [33]:
# Import Pydantic classes
from pydantic import BaseModel, Field


# Define the exact structure/schema we want from the LLM
class Customer(BaseModel):

    # LLM must return a string for name
    name: str = Field(
        description="Customer name"
    )

    # LLM must return an integer for age
    age: int = Field(
        description="Customer age"
    )


# Tell the LLM:
# "Whatever you answer, return it in Customer schema"
structured_llm = llm.with_structured_output(Customer)


# Ask the LLM
result = structured_llm.invoke(
    "I love cricket, which player is the best?"
)


# Print structured result
print(result)

name='Virat Kohli' age=35
